# OpenWebUI Trace PromptFoo Optimization Experiments

This notebook is a guided how-to for `trace_openwebui_dea_skeleton.py`. It helps non-expert users choose what to optimize, build bounded experiment commands, run them when explicitly enabled, and compare scores, parameters, and timings.

By default it is safe: it only builds and displays a plan. Set `RUN_OPENWEBUI_TRACE_EXPERIMENTS=1` to execute experiments.

## How To Choose What To Optimize

| Goal | Use | Configure |
|---|---|---|
| Model only, no tools | `optimize_target="model"` | `TARGET_MODEL_ID`, `SYSTEM_PROMPT`, `MODEL_PARAMS` |
| System prompt focused run | `PROFILE="model_system_prompt_only"` | `SYSTEM_PROMPT`; keep `MODEL_PARAMS` minimal |
| Sampling/length params | `PROFILE="model_sampling_params"` | `generation_temperature`, `generation_top_p`, `generation_max_tokens` |
| Provider/model-specific params | `PROFILE="model_extra_params"` | `model_extra_payload_json`, for example `frequency_penalty`, `think`, or provider-specific `reasoning` |
| Tool/pipe params | `optimize_target="tool"` | `TOOL_TARGET_MODEL_ID`, `TOOL_PARAMS` |
| PromptFoo Offline Process | `PROFILE="promptfoo_model_offline"` or `PROFILE="promptfoo_tool_offline"` | OpenWebUI generates first, PromptFoo scores the saved CSV |
| PromptFoo Online Process | `PROFILE="promptfoo_model_online"` or `PROFILE="promptfoo_tool_online"` | PromptFoo calls the OpenWebUI bridge live through its provider config |
| Compare methods | `PROFILE="method_matrix"` | Runs direct DEA, DEA judge, LLM judge, Offline Process, and Online Process examples |

`model_extra_payload_json` is intentionally configurable. It is forwarded to OpenWebUI/OpenAI-compatible payloads, but the target model/provider may ignore keys it does not support.

## Using Any PromptFoo Evaluation

DEA configs are examples, not requirements. The Trace harness can optimize an OpenWebUI model or tool using any PromptFoo config as long as the PromptFoo command writes a JSON result to `{result_json}`. That JSON can be PromptFoo's normal output, a red-team report adapted to a score JSON, or a custom wrapper with a top-level `score` or nested row records containing `score` and optional `namedScores`.

| User-facing process | Internal method | What happens | Typical PromptFoo config |
|---|---|---|---|
| Offline Process | `step2_promptfoo` | Trace calls OpenWebUI first, writes `candidate_answer` into a temporary CSV, then PromptFoo evaluates that CSV | `echo` provider or assertions over `{{candidate_answer}}` |
| Online Process | `step3_promptfoo` | Trace prepares rows, then PromptFoo calls the OpenWebUI bridge live through its provider config and evaluates returned output | HTTP provider calling `/generate` or another live provider |

For a custom CSV, provide at least one prompt column: `request_prompt`, `prompt`, `question`, `input`, `query`, or `instruction`. Extra columns such as `expected`, `reference`, `attack_category`, `policy`, or rubric fields are preserved for PromptFoo assertions. For JSON or JSONL datasets, set `BUILD_INPUT_CSV_CMD` to a command that creates the CSV path passed as `{output_csv}`.

## Objective Evidence Matrix

| Requirement | Current evidence | Runtime caveat |
|---|---|---|
| Optimize model parameters | `OpenWebUIModel`, model trainer tests, model notebook profiles | Live quality depends on target model availability and speed |
| Optimize tool parameters | `OpenWebUISummarizerAgent`, tool forwarding tests, tool notebook profiles | The OpenWebUI pipe/tool must consume the forwarded params |
| Offline Process | Trainer test covers bridge generation followed by PromptFoo on `step2_candidate_rows.csv` | Full live generation depends on OpenWebUI/backend health |
| Online Process | Trainer test covers prepared live rows and PromptFoo scoring artifacts | PromptFoo response transforms can hide bridge internals |
| Generic PromptFoo configs | PromptFoo result parsing accepts top-level scores or nested row scores | Red-team reports may need an adapter command to write `{result_json}` |
| Direct DEA with DEA judge | DEA guide test verifies `use_dea_judge=True` forwarding | Real judge latency/cost depends on configured judge backend |
| Trace and feedback | Step artifacts include summaries, traces, candidates, feedback, scores, params, durations | Online Process records PromptFoo command/runtime unless trace sidecars are added |
| Example tool models | The `summarizer---...` IDs are examples from this OpenWebUI setup | Replace them with your own model/tool IDs as needed |

# Test

In [64]:
from pathlib import Path
import os
import requests
from dotenv import load_dotenv

env_path = Path(r"C:\Users\maelle\Documents\document_embedding_analysis\scripts\promptfoo_openwebui_eval\.env")
load_dotenv(env_path, override=True)

print("ENV loaded:", env_path.exists())
print("DEA_REPO_PATH =", os.getenv("DEA_REPO_PATH"))
print("OPENWEBUI_BASE_URL =", os.getenv("OPENWEBUI_BASE_URL"))
print("OPENWEBUI_PIPE_MODEL =", os.getenv("OPENWEBUI_PIPE_MODEL"))
print("OPENAI_BASE_URL =", os.getenv("OPENAI_BASE_URL"))
print("DEA_JUDGE_MODEL =", os.getenv("DEA_JUDGE_MODEL"))
print("OWUI_BRIDGE_PORT =", os.getenv("OWUI_BRIDGE_PORT"))

ENV loaded: True
DEA_REPO_PATH = C:\Users\maelle\Documents\document_embedding_analysis
OPENWEBUI_BASE_URL = http://127.0.0.1:8002
OPENWEBUI_PIPE_MODEL = openai/owl-alpha
OPENAI_BASE_URL = https://openrouter.ai/api/v1
DEA_JUDGE_MODEL = openai/owl-alpha
OWUI_BRIDGE_PORT = 8003


In [65]:
base_url = os.getenv("OPENWEBUI_BASE_URL")
api_key = os.getenv("OPENWEBUI_API_KEY")

r = requests.get(
    f"{base_url}/api/models",
    headers={"Authorization": f"Bearer {api_key}"},
    timeout=30
)

print(r.status_code)
print(r.text[:1000])

200
{"data":[{"id":"llama-guard3:latest","name":"llama-guard3:latest","object":"model","created":1782473482,"owned_by":"ollama","ollama":{"name":"llama-guard3:latest","model":"llama-guard3:latest","modified_at":"2026-05-29T08:37:08.4595175+02:00","size":4920745186,"digest":"46f211c3d8662823ea4956d1a9e92b2924f84cbd0a0f88b64eaadb59fe58b99d","details":{"parent_model":"","format":"gguf","family":"llama","families":["llama"],"parameter_size":"8.0B","quantization_level":"Q4_K_M","context_length":131072,"embedding_length":4096},"capabilities":["completion"],"connection_type":"local","urls":[0]},"loaded":false,"connection_type":"local","tags":[],"actions":[],"filters":[]},{"id":"qwen3.6:35b","name":"qwen3.6:35b","object":"model","created":1782473482,"owned_by":"ollama","ollama":{"name":"qwen3.6:35b","model":"qwen3.6:35b","modified_at":"2026-05-21T16:26:37.5094148+02:00","size":23938333577,"digest":"07d35212591fc27746f0a317c975a6d68754fb38e9053d82e25f06057af28522","details":{"parent_model":"","

In [66]:
import requests

response = requests.post(
    "https://openrouter.ai/api/v1/chat/completions",
    headers={
        "Authorization": "Bearer sk-",
        "Content-Type": "application/json"
    },
    json={
        "model": "openrouter/owl-alpha",
        "messages": [{"role": "user", "content": "Dis OK"}],
        "max_tokens": 10
    }
)

print(f"Status: {response.status_code}")
print(f"Response: {response.text}")

Status: 200
Response: 
         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         

         
{"id":"gen-1782473484-switfVyGMUkJHS6JDqsr","object":"chat.completion","created":1782473484,"model":"openrout

In [79]:
openai_base = os.getenv("OPENAI_BASE_URL")
openai_key = os.getenv("OPENAI_API_KEY")
judge_model = os.getenv("DEA_JUDGE_MODEL")

payload = {
    "model": judge_model,
    "messages": [
        {"role": "user", "content": "Réponds seulement: OK"}
    ],
    "temperature": 0,
    "max_tokens": 20,
}

r = requests.post(
    f"{openai_base}/chat/completions",
    headers={
        "Authorization": f"Bearer {openai_key}",
        "Content-Type": "application/json",
    },
    json=payload,
    timeout=120,
)

print("status =", r.status_code)
print(r.text[:2000])

status = 200

         

         

         
{"id":"gen-1782466105-zh800EeQYL28SkPB3g46","object":"chat.completion","created":1782466105,"model":"openrouter/owl-alpha","provider":"Stealth","system_fingerprint":null,"service_tier":null,"choices":[{"index":0,"logprobs":null,"finish_reason":"stop","native_finish_reason":"stop","message":{"role":"assistant","content":"OK","refusal":null,"reasoning":null}}],"usage":{"prompt_tokens":108,"completion_tokens":2,"total_tokens":110,"cost":0,"is_byok":false,"prompt_tokens_details":{"cached_tokens":0,"cache_write_tokens":0,"audio_tokens":0,"video_tokens":0},"cost_details":{"upstream_inference_cost":0,"upstream_inference_prompt_cost":0,"upstream_inference_completions_cost":0},"completion_tokens_details":{"reasoning_tokens":0,"image_tokens":0,"audio_tokens":0}}}


In [89]:
import requests

response = requests.post(
    "http://127.0.0.1:8003/v1/chat/completions",
    headers={
        "Content-Type": "application/json",
        "Authorization": "Bearer dummy"
    },
    json={
        "model": "openrouter/owl-alpha",
        "messages": [{"role": "user", "content": "Dis juste 'OK'"}],
        "stream": False
    }
)

print(f"Status: {response.status_code}")
print(f"Response: {response.text[:500]}")

Status: 200
Response: {"id":"chatcmpl-d7cfed353918479398ea129f46a94dcc","object":"chat.completion","created":1782466401,"model":"openrouter/owl-alpha","choices":[{"index":0,"message":{"role":"assistant","content":"OK"},"finish_reason":"stop"}]}


# Fonction de config

In [67]:
# Charger le fichier .env
try:
    from dotenv import load_dotenv
    load_dotenv()  # Charge automatiquement le .env du répertoire courant
    print("✅ .env chargé")
except ImportError:
    print("⚠️ python-dotenv non installé. Installe-le avec: pip install python-dotenv")

print("=== Environment after notebook configuration cell ===")
print("OPENAI_MODEL =", os.environ.get("OPENAI_MODEL"))
print("DEA_JUDGE_MODEL =", os.environ.get("DEA_JUDGE_MODEL"))
print("TRACE_LITELLM_MODEL =", os.environ.get("TRACE_LITELLM_MODEL"))
print("TRACE_CUSTOMLLM_MODEL =", os.environ.get("TRACE_CUSTOMLLM_MODEL"))
print("DEFAULT_LITELLM_MODEL =", os.environ.get("DEFAULT_LITELLM_MODEL"))
print("OPENWEBUI_PIPE_MODEL =", os.environ.get("OPENWEBUI_PIPE_MODEL"))
print("OPENWEBUI_DEFAULT_SUMMARIZER_MODEL_ID =", os.environ.get("OPENWEBUI_DEFAULT_SUMMARIZER_MODEL_ID"))


# Charger le .env AVANT tout le reste
from dotenv import load_dotenv
load_dotenv()  # Charge automatiquement le .env du répertoire courant

# Vérification
import os
print("✅ Variables critiques chargées :")
for var in ['OPENAI_API_KEY', 'OPENROUTER_API_KEY', 'OPENAI_BASE_URL', 'OPENAI_MODEL']:
    val = os.environ.get(var, '')
    if val:
        print(f"  ✓ {var} = {val[:30]}...")
    else:
        print(f"  ✗ {var} = NON DÉFINI !")


✅ .env chargé
=== Environment after notebook configuration cell ===
OPENAI_MODEL = openai/owl-alpha
DEA_JUDGE_MODEL = openai/owl-alpha
TRACE_LITELLM_MODEL = openai/owl-alpha
TRACE_CUSTOMLLM_MODEL = openai/owl-alpha
DEFAULT_LITELLM_MODEL = openai/owl-alpha
OPENWEBUI_PIPE_MODEL = openai/owl-alpha
OPENWEBUI_DEFAULT_SUMMARIZER_MODEL_ID = openai/owl-alpha
✅ Variables critiques chargées :
  ✓ OPENAI_API_KEY = sk-or-v1-fa23c161925570046aded...
  ✓ OPENROUTER_API_KEY = OPENAI_ENDPOINT_API_KEY...
  ✓ OPENAI_BASE_URL = https://openrouter.ai/api/v1...
  ✓ OPENAI_MODEL = openai/owl-alpha...


In [88]:
# les imports
from __future__ import annotations

import json
import os
import shlex
import subprocess
import time
from datetime import datetime
from pathlib import Path
from typing import Any

try:
    import pandas as pd
except Exception:
    pd = None

from IPython.display import Markdown, display



# =============================================================================
# Fonction d'initialisation/ mise en forme
# =============================================================================

# cherche le dépôt 
def find_repo_root(start: Path) -> Path:
    """Return the repository root containing the Trace optimization script."""
    for candidate in [start, *start.parents]:
        if (candidate / 'scripts/promptfoo_openwebui_eval/trace_openwebui_dea_skeleton.py').exists():
            return candidate
    raise RuntimeError(f'Could not find repo root from {start}')


#  cherche si des variables d'environnement on été défini sous forme booléenne ou JSON
def env_bool(name: str, default: bool = False) -> bool:
    """Read a boolean environment flag using common truthy strings."""
    raw = os.environ.get(name, '').strip().lower()
    if not raw:
        return default
    return raw in {'1', 'true', 'yes', 'on'}
def env_json(name: str, default: dict[str, Any]) -> dict[str, Any]:
    """Read a JSON object from an environment variable or return defaults."""
    raw = os.environ.get(name, '').strip()
    if not raw:
        return dict(default)
    parsed = json.loads(raw)
    if not isinstance(parsed, dict):
        raise ValueError(f'{name} must contain a JSON object')
    return parsed


# Convertit un chemin absolu en chemin relatif au dépôt quand c'est possible
def rel(path: Path | str) -> str:
    """Render a path relative to the repository when possible."""
    path = Path(path)
    try:
        return str(path.resolve().relative_to(REPO_ROOT))
    except Exception:
        return str(path)


# affiche un dictionnaire comme tableau pour rendre les traces plus lissible 
def show_table(rows: list[dict[str, Any]], title: str = '') -> None:
    """Display rows as a DataFrame, falling back to JSON markdown."""
    if title:
        display(Markdown(f'### {title}'))
    if pd is not None:
        display(pd.DataFrame(rows))
    else:
        display(Markdown('```json\n' + json.dumps(rows, indent=2, ensure_ascii=False) + '\n```'))



# =============================================================================
# Configuration générale du projet -> emplacement 
# =============================================================================

# Racine du dépôt.
# Normalement détectée automatiquement.
REPO_ROOT = Path(os.environ['TRACE_REPO_ROOT']).resolve() if os.environ.get('TRACE_REPO_ROOT') else find_repo_root(Path.cwd())

# du plus haut au plus bas: dossier promptfoo_openwebui_eval
#                         : fichier trace_openwebui_dea_skeleton
#                         : nom de dataset donné au dataset d'optimisation
#                         : dataset d'entrainement
#                         : bridge
#                         : Promptfoo
BUNDLE = REPO_ROOT / 'scripts/promptfoo_openwebui_eval'
SCRIPT = BUNDLE / 'trace_openwebui_dea_skeleton.py'
DATASET_NAME = os.environ.get('TRACE_DATASET_NAME', 'bigsurvey')
INPUT_CSV = Path(os.environ.get('TRACE_INPUT_CSV', BUNDLE / 'datasets' / DATASET_NAME / 'step2_input.csv')).resolve()
BRIDGE_URL = os.environ.get('TRACE_BRIDGE_URL', 'http://127.0.0.1:8003/generate')
PROMPTFOO_WORKDIR = Path(os.environ.get('TRACE_PROMPTFOO_WORKDIR', BUNDLE)).resolve()



# =============================================================================
# Configuration générale du projet -> résultat
# =============================================================================

# Identifiant du run.
RUN_ID = os.environ.get('TRACE_EXPERIMENT_RUN_ID') or datetime.now().strftime('%Y%m%d_%H%M%S')

# nom et emplacement des résultats
RESULTS_DIR = BUNDLE / 'results' / 'trace_experiments' / RUN_ID
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

#  Test si le notebook lance les commandes 
RUN_LIVE = env_bool('RUN_OPENWEBUI_TRACE_EXPERIMENTS', True)

# Commande utilisée pour lancer l'évaluation Promptfoo et écrire le JSON de résultats
PROMPTFOO_EVAL_CMD = os.environ.get('TRACE_PROMPTFOO_EVAL_CMD', 'promptfoo eval -c {config} -t {csv} --output {result_json} --no-progress-bar')

# ?????????
BUILD_INPUT_CSV_CMD = os.environ.get('TRACE_BUILD_INPUT_CSV_CMD', '').strip()

# Si True, régénère le CSV même s'il existe déjà
FORCE_BUILD_INPUT_CSV = env_bool('TRACE_FORCE_BUILD_INPUT_CSV', False)


# Initialiser l'experience/ configuration globale

In [89]:
# =============================================================================
# Configuration générale du projet -> paramètre
# =============================================================================

# Profil d'expérience utilisé
PROFILE = "model_system_prompt_only"

# Modèles
TARGET_MODEL_ID = "openrouter/owl-alpha"
TOOL_TARGET_MODEL_ID = "openrouter/owl-alpha"
JUDGE_MODEL = "openrouter/owl-alpha"

# Bridge
BRIDGE_URL = "http://127.0.0.1:8003/generate"

# Paramètres d'optimisation
MAX_ROWS = 1          # nombre de lignes du CSV utilisées
BATCH_SIZE = 2        # nombre de lignes par feedback
NUM_EPOCHS = 3        # nombre de passages
OPTIMIZER_MODE = "optoprime" # noop ou optoptime
OPTIMIZER_MAX_TOKENS = 256

# Prompt système initial
PROMPT_REF = (
    "You are a careful literature-review assistant. "
    "Use only the provided source documents. "
    "Do not invent unsupported claims. "
    "Keep citations/source references faithful to the provided context."
)


# Paramètres initiaux du modèle
MODEL_PARAMS = env_json('TRACE_MODEL_PARAMS_JSON', {'generation_temperature': 0.2, 'generation_top_p': 0.95, 'generation_max_tokens': 512})


# Affichage de la config

In [90]:
# =============================================================================
# Pour l'affichage 
# =============================================================================

# type d'experience = profile
PROFILE = os.environ.get('TRACE_EXPERIMENT_PROFILE', PROFILE)

# Modèle
TARGET_MODEL_ID = os.environ.get('TRACE_TARGET_MODEL_ID', TARGET_MODEL_ID)
TOOL_TARGET_MODEL_ID = os.environ.get('TRACE_TOOL_TARGET_MODEL_ID', TOOL_TARGET_MODEL_ID)

# param opti
BATCH_SIZE = int(os.environ.get('TRACE_BATCH_SIZE', BATCH_SIZE)) 
NUM_EPOCHS = int(os.environ.get('TRACE_NUM_EPOCHS', NUM_EPOCHS)) 
MAX_ROWS = int(os.environ.get('TRACE_MAX_ROWS', MAX_ROWS))  

# param init du modèle
SYSTEM_PROMPT = os.environ.get('TRACE_SYSTEM_PROMPT', PROMPT_REF)
MODEL_EXTRA_PAYLOAD = env_json('TRACE_MODEL_EXTRA_PAYLOAD_JSON', {'frequency_penalty': 0.0, 'think': False})
TOOL_PARAMS = env_json('TRACE_TOOL_PARAMS_JSON', {'algorithm': 'kohaku', 'target_length': 'long', 'structure': 'thematic'})

# appel promptfoo 
PROMPTFOO_OFFLINE_CONFIG = os.environ.get('TRACE_PROMPTFOO_OFFLINE_CONFIG', f'{DATASET_NAME}.step2.dea.yaml') # OpenWebUI génère les réponses, puis Promptfoo les évalue
PROMPTFOO_ONLINE_CONFIG = os.environ.get('TRACE_PROMPTFOO_ONLINE_CONFIG', f'{DATASET_NAME}.step3.dea.yaml')   # Online = q° généré et évalué par Promptfoo 



# =============================================================================
# Construction de l'experience
# =============================================================================

résum = {
    'repo_root': '.',
    'bundle': rel(BUNDLE),
    'dataset_name': DATASET_NAME,
    'input_csv': rel(INPUT_CSV),
    'results_dir': rel(RESULTS_DIR),
    'run_live': RUN_LIVE,
    'profile': PROFILE,
    'batch_size': BATCH_SIZE,
    'num_epochs': NUM_EPOCHS,
    'target_model_id': TARGET_MODEL_ID,
    'tool_target_model_id': TOOL_TARGET_MODEL_ID,
    'promptfoo_workdir': rel(PROMPTFOO_WORKDIR),
    'promptfoo_offline_config': PROMPTFOO_OFFLINE_CONFIG,
    'promptfoo_online_config': PROMPTFOO_ONLINE_CONFIG,
    'has_build_input_csv_cmd': bool(BUILD_INPUT_CSV_CMD),
}

# affichage  
show_table([résum], 'Expérience en cours')

### Expérience en cours

,repo_root,bundle,dataset_name,input_csv,results_dir,run_live,profile,batch_size,num_epochs,target_model_id,tool_target_model_id,promptfoo_workdir,promptfoo_offline_config,promptfoo_online_config,has_build_input_csv_cmd
0,.,scripts\promptfoo_openwebui_eval,bigsurvey,scripts\promptfoo_openwebui_eval\datasets\bigs...,scripts\promptfoo_openwebui_eval\results\trace...,True,model_system_prompt_only,2,3,openrouter/owl-alpha,openrouter/owl-alpha,scripts\promptfoo_openwebui_eval,bigsurvey.step2.dea.yaml,bigsurvey.step3.dea.yaml,False


# Recap méthode dispo 

In [91]:
# =============================================================================
# # Récap des moyens d'optimiser possible (à titre informatif)
# =============================================================================

# Liste des outils/modèles OpenWebUI utilisables pour les tests
tool_models = [
    {'algorithm': 'dtcrs', 'target_model_id': 'summarizer---dtcrs', 'note': 'example tool/pipe model id'},
    {'algorithm': 'kohaku', 'target_model_id': 'summarizer---kohaku', 'note': 'example tool/pipe model id'},
    {'algorithm': 'kohaku_openrouter', 'target_model_id': 'summarizer---kohaku-OR', 'note': 'example OpenRouter-backed pipe id'},
    {'algorithm': 'lightrag', 'target_model_id': 'summarizer---lightrag', 'note': 'example tool/pipe model id'},
    {'algorithm': 'raptor', 'target_model_id': 'summarizer---raptor', 'note': 'example tool/pipe model id'},
]


# Liste méthode d'opti (Optimization methods and process names)
PROCESS_TO_METHOD = {'offline': 'step2_promptfoo', 'online': 'step3_promptfoo'} # Associe le nom du processus à la méthode appelée par le scripte
PROCESS_LABELS = {
    'offline': 'Offline Process (step2_promptfoo)',
    'online': 'Online Process (step3_promptfoo)',
}
method_matrix = [
    {'label': 'Direct DEA', 'method': 'direct_dea', 'generation': 'bridge', 'evaluation': 'native DEA', 'trace': 'bridge trace + total runtime', 'cost': 'medium'},
    {'label': 'Direct DEA judge', 'method': 'direct_dea_judge', 'generation': 'bridge', 'evaluation': 'native DEA + DEA LLM judge', 'trace': 'bridge trace + judge runtime', 'cost': 'high'},
    {'label': 'Direct LLM judge', 'method': 'direct_llm_judge', 'generation': 'bridge', 'evaluation': 'custom JSON LLM judge', 'trace': 'bridge trace + judge runtime', 'cost': 'medium/high'},
    {'label': PROCESS_LABELS['offline'], 'method': PROCESS_TO_METHOD['offline'], 'generation': 'bridge first', 'evaluation': 'PromptFoo scores saved CSV', 'trace': 'bridge trace + PromptFoo artifacts', 'cost': 'high'},
    {'label': PROCESS_LABELS['online'], 'method': PROCESS_TO_METHOD['online'], 'generation': 'PromptFoo live bridge provider', 'evaluation': 'PromptFoo live scoring', 'trace': 'PromptFoo artifacts + command runtime', 'cost': 'high'},
]


# Liste des types d'experiences (Recipe profiles)
optimization_recipes = [
    {'profile': 'model_system_prompt_only', 'target': 'model', 'configure': 'SYSTEM_PROMPT', 'notes': 'Keep MODEL_PARAMS minimal; useful for prompt-focused comparisons.'},
    {'profile': 'model_sampling_params', 'target': 'model', 'configure': 'MODEL_PARAMS', 'notes': 'Uses temperature/top_p/max_tokens convenience params.'},
    {'profile': 'model_extra_params', 'target': 'model', 'configure': 'MODEL_EXTRA_PAYLOAD', 'notes': 'For provider-specific params such as frequency_penalty or think.'},
    {'profile': 'tool_kohaku_example', 'target': 'tool', 'configure': 'TOOL_TARGET_MODEL_ID and TOOL_PARAMS', 'notes': 'Example only; depends on your OpenWebUI tool setup.'},
    {'profile': 'tool_algorithm_grid', 'target': 'tool', 'configure': 'tool_models', 'notes': 'Compares example algorithm-specific pipes.'},
    {'profile': 'promptfoo_model_offline', 'target': 'model', 'configure': 'PROMPTFOO_OFFLINE_CONFIG', 'notes': 'Offline Process: Trace generates answers, PromptFoo scores saved CSV.'},
    {'profile': 'promptfoo_model_online', 'target': 'model', 'configure': 'PROMPTFOO_ONLINE_CONFIG', 'notes': 'Online Process: PromptFoo calls the bridge live.'},
    {'profile': 'promptfoo_tool_offline', 'target': 'tool', 'configure': 'PROMPTFOO_OFFLINE_CONFIG', 'notes': 'Offline Process for a tool/pipe.'},
    {'profile': 'promptfoo_tool_online', 'target': 'tool', 'configure': 'PROMPTFOO_ONLINE_CONFIG', 'notes': 'Online Process for a tool/pipe.'},
    {'profile': 'offline_calibration', 'target': 'model + tool', 'configure': 'none', 'notes': 'Fast fake PromptFoo score, real Trace trainer artifacts, no OpenWebUI call.'},
]


# affichage
show_table(tool_models, 'Outil/ modèle openWeb UI')
show_table(method_matrix, 'Méthode optimisation')
show_table(optimization_recipes, 'Catalogue type experience')

### Outil/ modèle openWeb UI

,algorithm,target_model_id,note
0,dtcrs,summarizer---dtcrs,example tool/pipe model id
1,kohaku,summarizer---kohaku,example tool/pipe model id
2,kohaku_openrouter,summarizer---kohaku-OR,example OpenRouter-backed pipe id
3,lightrag,summarizer---lightrag,example tool/pipe model id
4,raptor,summarizer---raptor,example tool/pipe model id


### Méthode optimisation

,label,method,generation,evaluation,trace,cost
0,Direct DEA,direct_dea,bridge,native DEA,bridge trace + total runtime,medium
1,Direct DEA judge,direct_dea_judge,bridge,native DEA + DEA LLM judge,bridge trace + judge runtime,high
2,Direct LLM judge,direct_llm_judge,bridge,custom JSON LLM judge,bridge trace + judge runtime,medium/high
3,Offline Process (step2_promptfoo),step2_promptfoo,bridge first,PromptFoo scores saved CSV,bridge trace + PromptFoo artifacts,high
4,Online Process (step3_promptfoo),step3_promptfoo,PromptFoo live bridge provider,PromptFoo live scoring,PromptFoo artifacts + command runtime,high


### Catalogue type experience

,profile,target,configure,notes
0,model_system_prompt_only,model,SYSTEM_PROMPT,Keep MODEL_PARAMS minimal; useful for prompt-f...
1,model_sampling_params,model,MODEL_PARAMS,Uses temperature/top_p/max_tokens convenience ...
2,model_extra_params,model,MODEL_EXTRA_PAYLOAD,For provider-specific params such as frequency...
3,tool_kohaku_example,tool,TOOL_TARGET_MODEL_ID and TOOL_PARAMS,Example only; depends on your OpenWebUI tool s...
4,tool_algorithm_grid,tool,tool_models,Compares example algorithm-specific pipes.
5,promptfoo_model_offline,model,PROMPTFOO_OFFLINE_CONFIG,"Offline Process: Trace generates answers, Prom..."
6,promptfoo_model_online,model,PROMPTFOO_ONLINE_CONFIG,Online Process: PromptFoo calls the bridge live.
7,promptfoo_tool_offline,tool,PROMPTFOO_OFFLINE_CONFIG,Offline Process for a tool/pipe.
8,promptfoo_tool_online,tool,PROMPTFOO_ONLINE_CONFIG,Online Process for a tool/pipe.
9,offline_calibration,model + tool,none,"Fast fake PromptFoo score, real Trace trainer ..."


# Convertion config en experience 

In [92]:
# Appel Promptfoo : test pipeline sans vrai appel LLM
def fake_promptfoo_command() -> str:
    """Return a deterministic PromptFoo command for offline plumbing checks."""
    return (
        "python -c \"import json,os,pathlib,sys; "
        "path=pathlib.Path(sys.argv[1]); root=os.environ.get('DEA_REPO_ROOT'); "
        "path=path if path.is_absolute() or not root else pathlib.Path(root) / path; "
        "path.parent.mkdir(parents=True, exist_ok=True); "
        "payload=dict(score=0.73, namedScores=dict(promptfoo_score=0.73)); "
        "path.write_text(json.dumps(payload), encoding='utf-8')\" {result_json}"
    )


# Ajout de param si besoin
def param_sup(params: dict[str, Any], extra_payload: dict[str, Any]) -> dict[str, Any]:
    """Return model params with provider-specific payload keys attached."""
    merged = dict(params)
    merged['model_extra_payload_json'] = dict(extra_payload)
    return merged

# Select fichier appel/config promptfoo ; 2 types: off/on -line
def mode_line(process: str) -> str:
    """Return the configured PromptFoo config for an Offline or Online Process."""
    if process == 'offline':
        return PROMPTFOO_OFFLINE_CONFIG
    if process == 'online':
        return PROMPTFOO_ONLINE_CONFIG
    raise ValueError(f'Unknown process {process!r}')


# On rempli le fichier avec la config  
def config(
    name: str,
    *,
    process: str,
    optimize_target: str,
    target_model_id: str,
    system_prompt: str | None = None,
    model_params: dict[str, Any] | None = None,
    tool_params: dict[str, Any] | None = None,
    promptfoo_config: str | None = None,
    promptfoo_eval_cmd: str | None = None,
    optimizer_mode: str = 'noop',
) -> dict[str, Any]:
    """Build one generic PromptFoo Offline/Online Process experiment."""
    return {
        'name': name,
        'process': process,
        'process_label': PROCESS_LABELS[process],
        'method': PROCESS_TO_METHOD[process],
        'optimize_target': optimize_target,
        'target_model_id': target_model_id,
        'system_prompt': system_prompt or SYSTEM_PROMPT,
        'model_params': model_params or MODEL_PARAMS,
        'tool_params': tool_params or TOOL_PARAMS,
        'promptfoo_config': promptfoo_config or mode_line(process),
        'promptfoo_workdir': rel(PROMPTFOO_WORKDIR),
        'promptfoo_eval_cmd': promptfoo_eval_cmd or PROMPTFOO_EVAL_CMD,
        'optimizer_mode': optimizer_mode,
    }


# A chaque profil est associé une ou plusieurs experiences 
def exp_ass(profile: str) -> list[dict[str, Any]]:
    """Build experiment definitions for the selected recipe profile."""
    profiles: dict[str, list[dict[str, Any]]] = {
# prompt systeme
        'model_system_prompt_only': [
            {
                'name': 'optim_model_openrouter_prompt_only',
                'method': 'direct_llm_judge',
                'process_label': 'Direct LLM judge',
                'optimize_target': 'model',
                'target_model_id': TARGET_MODEL_ID,
                'max_rows': MAX_ROWS,
                'batch_size': BATCH_SIZE,
                'num_epochs': NUM_EPOCHS,
                'optimizer_mode': OPTIMIZER_MODE,
                'optimizer_max_tokens': OPTIMIZER_MAX_TOKENS,
                'bridge_url': BRIDGE_URL,
                'system_prompt': SYSTEM_PROMPT,
                'model_params': MODEL_PARAMS,
                'model_prompt_mode': 'both',
                'judge_model': JUDGE_MODEL,
            }
        ],
# paramètre de base (temp, topk...)
        'model_sampling_params': [
            {
                'name': 'model_sampling_params',
                'method': 'direct_dea',
                'process_label': 'Direct DEA',
                'optimize_target': 'model',
                'target_model_id': TARGET_MODEL_ID,
                'system_prompt': SYSTEM_PROMPT,
                'model_params': MODEL_PARAMS,
            }
        ],
# paramètre suplémentaire 
        'model_extra_params': [
            {
                'name': 'model_extra_params',
                'method': 'direct_dea',
                'process_label': 'Direct DEA',
                'optimize_target': 'model',
                'target_model_id': TARGET_MODEL_ID,
                'system_prompt': SYSTEM_PROMPT,
                'model_params': param_sup(MODEL_PARAMS, MODEL_EXTRA_PAYLOAD),
            }
        ],
# tool
        'tool_kohaku_example': [
            {
                'name': 'tool_kohaku_example',
                'method': 'direct_dea',
                'process_label': 'Direct DEA',
                'optimize_target': 'tool',
                'target_model_id': TOOL_TARGET_MODEL_ID,
                'tool_params': TOOL_PARAMS,
            }
        ],
# tool    
        'tool_algorithm_grid': [
            {
                'name': f"tool_{item['algorithm']}",
                'method': 'direct_dea',
                'process_label': 'Direct DEA',
                'optimize_target': 'tool',
                'target_model_id': item['target_model_id'],
                'tool_params': {'algorithm': item['algorithm'].replace('_openrouter', ''), 'target_length': TOOL_PARAMS.get('target_length', 'long'), 'structure': TOOL_PARAMS.get('structure', 'thematic')},
            }
            for item in tool_models
        ],
# mode offline ou online
        'promptfoo_model_offline': [
            config('promptfoo_model_offline', process='offline', optimize_target='model', target_model_id=TARGET_MODEL_ID, system_prompt=SYSTEM_PROMPT, model_params=MODEL_PARAMS),
        ],
        'promptfoo_model_online': [
            config('promptfoo_model_online', process='online', optimize_target='model', target_model_id=TARGET_MODEL_ID, system_prompt=SYSTEM_PROMPT, model_params=MODEL_PARAMS),
        ],
        'promptfoo_tool_offline': [
            config('promptfoo_tool_offline', process='offline', optimize_target='tool', target_model_id=TOOL_TARGET_MODEL_ID, tool_params=TOOL_PARAMS),
        ],
        'promptfoo_tool_online': [
            config('promptfoo_tool_online', process='online', optimize_target='tool', target_model_id=TOOL_TARGET_MODEL_ID, tool_params=TOOL_PARAMS),
        ],
# 
        'method_matrix': [
            {
                'name': f"method_{item['method']}",
                'method': item['method'],
                'process_label': item['label'],
                'optimize_target': 'tool',
                'target_model_id': TOOL_TARGET_MODEL_ID,
                'tool_params': TOOL_PARAMS,
            }
            if item['method'] not in {PROCESS_TO_METHOD['offline'], PROCESS_TO_METHOD['online']}
            else config(
                f"method_{item['method']}",
                process='offline' if item['method'] == PROCESS_TO_METHOD['offline'] else 'online',
                optimize_target='tool',
                target_model_id=TOOL_TARGET_MODEL_ID,
                tool_params=TOOL_PARAMS,
            )
            for item in method_matrix
        ],
        'offline_calibration': [
            config(
                'offline_model_calibration',
                process='online',
                optimize_target='model',
                target_model_id=TARGET_MODEL_ID,
                system_prompt='Calibration run only.',
                model_params={'generation_temperature': 0, 'generation_max_tokens': 64, 'model_extra_payload_json': {'frequency_penalty': 0.0, 'think': False}},
                promptfoo_eval_cmd=fake_promptfoo_command(),
                optimizer_mode='optoprime',
            ),
            config(
                'offline_tool_calibration',
                process='online',
                optimize_target='tool',
                target_model_id=TOOL_TARGET_MODEL_ID,
                tool_params={'algorithm': TOOL_PARAMS.get('algorithm', 'kohaku'), 'target_length': 'short', 'structure': TOOL_PARAMS.get('structure', 'thematic')},
                promptfoo_eval_cmd=fake_promptfoo_command(),
                optimizer_mode='optoprime',
            ),
        ],
    }
    if profile not in profiles:
        raise ValueError(f'Unknown profile {profile!r}. Choose one of: {sorted(profiles)}')
    return profiles[profile]


# appel fct
experiments = exp_ass(PROFILE)

# affichage
show_table(experiments, f'Selected experiments: {PROFILE}')

### Selected experiments: model_system_prompt_only

,name,method,process_label,optimize_target,target_model_id,max_rows,batch_size,num_epochs,optimizer_mode,optimizer_max_tokens,bridge_url,system_prompt,model_params,model_prompt_mode,judge_model
0,optim_model_openrouter_prompt_only,direct_llm_judge,Direct LLM judge,model,openrouter/owl-alpha,1,2,3,optoprime,256,http://127.0.0.1:8003/generate,You are a careful literature-review assistant....,"{'generation_temperature': 0.2, 'generation_to...",both,openrouter/owl-alpha


# Convertion expérience en commande 

In [93]:
def base_command(exp: dict[str, Any]) -> list[str]:
    """Convert one experiment definition into a Trace CLI command."""
    # Définir nom fichier de sortie 
    output_json = RESULTS_DIR / f"{exp['name']}.json"
    artifact_dir = RESULTS_DIR / exp['name']
    # Arguments communs à toutes les expériences
    args = [
        'python', rel(SCRIPT),
        '--input-csv', rel(INPUT_CSV),
        '--max-rows', str(exp.get('max_rows', MAX_ROWS)),
        '--batch-size', str(exp.get('batch_size', BATCH_SIZE)),
        '--num-epochs', str(exp.get('num_epochs', NUM_EPOCHS)),
        '--optimization-method', exp['method'],
        '--optimize-target', exp['optimize_target'],
        '--target-model-id', exp['target_model_id'],
        '--bridge-url', exp.get('bridge_url', BRIDGE_URL),
        '--bridge-timeout-seconds', str(exp.get('bridge_timeout_seconds', 600)),
        '--artifact-dir', rel(artifact_dir),
        '--output-json', rel(output_json),
        '--optimizer-mode', exp.get('optimizer_mode', 'noop'),
        '--optimizer-max-tokens', str(exp.get('optimizer_max_tokens', 512)),
    ]

    # construction du csv de l'experience
    build_cmd = exp.get('build_input_csv_cmd', BUILD_INPUT_CSV_CMD)
    if build_cmd:
        args += ['--build-input-csv-cmd', build_cmd]
    if exp.get('force_build_input_csv', FORCE_BUILD_INPUT_CSV):
        args += ['--force-build-input-csv']
        
    # Cas 1 : optimisation d'un outil/pipe OpenWebUI
    if exp['optimize_target'] == 'tool':
        args += ['--initial-tool-params-json', json.dumps(exp.get('tool_params', {}), ensure_ascii=False)]

    # Cas 2 : optimisation d'un modèle
    else:
        args += [
            '--initial-system-prompt', exp.get('system_prompt', SYSTEM_PROMPT),
            '--initial-model-params-json', json.dumps(exp.get('model_params', MODEL_PARAMS), ensure_ascii=False),
        ]
        
        if exp.get('model_prompt_mode'):
            args += ['--model-prompt-mode', exp['model_prompt_mode']]

    # utilisation d'un modèle juge promptfoo
    if exp['method'] in set(PROCESS_TO_METHOD.values()):
        args += [
            '--promptfoo-config', exp.get('promptfoo_config', mode_line(exp.get('process', 'offline'))),
            '--promptfoo-workdir', exp.get('promptfoo_workdir', rel(PROMPTFOO_WORKDIR)),
            '--promptfoo-eval-cmd', exp.get('promptfoo_eval_cmd', PROMPTFOO_EVAL_CMD),
        ]

    # utilisation d'un modèle juge
    if exp['method'] == 'direct_llm_judge':
        args += [
            '--judge-base-url', exp.get('judge_base_url', 'http://127.0.0.1:8003/v1'),
            '--judge-api-key', exp.get('judge_api_key', 'dummy'),
            '--judge-model', exp.get('judge_model', 'openrouter/owl-alpha'),
            '--judge-timeout-seconds', str(exp.get('judge_timeout_seconds', 180)),
            '--judge-extra-body-json', json.dumps(exp.get('judge_extra_body', {'max_tokens': 192}), ensure_ascii=False),
            '--judge-system-prompt', exp.get('judge_system_prompt', 'Return compact strict JSON only.'),
        ]
    return args

# Lancement de l'experience 

In [94]:
# Les résultats
def summarize_output(path: Path) -> dict[str, Any]:
    """Extract score, timing, and parameter columns from a Trace result JSON."""
    if not path.exists():
        return {}
    payload = json.loads(path.read_text(encoding='utf-8'))

    def search_keys(obj, words):
        if isinstance(obj, dict):
            for k, v in obj.items():
                if any(w in k.lower() for w in words):
                    print("\nKEY:", k)
                    print(str(v)[:1500])
                search_keys(v, words)
        elif isinstance(obj, list):
            for x in obj:
                search_keys(x, words)
    
    search_keys(payload, ["update", "proposal", "propose", "optimizer", "gradient", "prompt"])

    best = payload.get('best_weighted') or {}
    scores = best.get('mean_scores') or {}
    state = best.get('current_state') or {}
    return {
        'best_scalar_objective': best.get('scalar_objective'),
        'score': scores.get('score'),
        'dea_composite': scores.get('dea_composite'),
        'plan': scores.get('plan'),
        'content': scores.get('content'),
        'resources': scores.get('resources'),
        'length_alignment': scores.get('length_alignment'),
        'execution_duration_s': scores.get('execution_duration_s'),
        'promptfoo_duration_s': scores.get('promptfoo_duration_s'),
        'llm_calls': scores.get('llm_calls'),
        'tool_calls': scores.get('tool_calls'),
        'batch_size': best.get('batch_size'),
        'optimizer_mode': best.get('optimizer_mode'),
        'param_json': state.get('param_json'),
        'model_param_json': state.get('model_param_json'),
        'tool_param_json': state.get('tool_param_json'),
        'system_prompt_chars': len(state.get('system_prompt') or ''),
    }


# Parcours de toutes les expériences construites précédemment 
# Purpose: optionally execute selected experiments and write score/parameter/timing result tables.
results = []
for exp in experiments:
    cmd = base_command(exp)
    output_json = RESULTS_DIR / f"{exp['name']}.json"
    row = {
        'name': exp['name'],
        'process': exp.get('process_label', exp.get('method')),
        'method': exp['method'],
        'target_model_id': exp['target_model_id'],
        'status': 'planned',
    }
    if RUN_LIVE:
        t0 = time.time()
    
        env = dict(
            os.environ,
            DEA_REPO_ROOT=str(REPO_ROOT),
            DEA_REPO_PATH=str(REPO_ROOT),
        )

        proc = subprocess.run(
            cmd, 
            cwd=str(REPO_ROOT), 
            env=env, 
            text=True, 
            capture_output=True, 
            timeout=int(os.environ.get('TRACE_EXPERIMENT_TIMEOUT_SECONDS', '7200'))
        )
        
        stdout_path = RESULTS_DIR / f"{exp['name']}.stdout.txt"
        stderr_path = RESULTS_DIR / f"{exp['name']}.stderr.txt"
        safe_stdout = proc.stdout.replace(str(REPO_ROOT), '.')
        safe_stderr = proc.stderr.replace(str(REPO_ROOT), '.')
        stdout_path.write_text(safe_stdout, encoding='utf-8')
        stderr_path.write_text(safe_stderr, encoding='utf-8')

        # info dans le tableau 
        row.update({
            'status': 'ok' if proc.returncode == 0 else 'failed',
            'returncode': proc.returncode,
            'wall_s': time.time() - t0,
            'log_stdout': rel(stdout_path),
            'log_stderr': rel(stderr_path),
            'error_tail': safe_stderr[-1000:] if proc.returncode else '',
        })
        row.update(summarize_output(output_json))
    
    results.append(row)
    show_table([row], f"Result: {exp['name']}")

# affichage dans tab
show_table(results, 'Summary results')

# les fichiers info avec résult
if pd is not None:
    pd.DataFrame(results).to_csv(RESULTS_DIR / 'experiment_results.csv', index=False)
else:
    (RESULTS_DIR / 'experiment_results.json').write_text(
        json.dumps(results, indent=2, ensure_ascii=False), 
        encoding='utf-8'
    )


KEY: optimizer_mode
optoprime

KEY: system_prompt
You are a careful literature-review assistant. Use only the provided source documents. Do not invent unsupported claims. Keep citations/source references faithful to the provided context.

KEY: optimizer_mode
optoprime

KEY: system_prompt
You are a careful literature-review assistant. Use only the provided source documents. Do not invent unsupported claims. Keep citations/source references faithful to the provided context.

KEY: optimizer_mode
optoprime

KEY: system_prompt
You are a careful literature-review assistant. Use only the provided source documents. Do not invent unsupported claims. Keep citations/source references faithful to the provided context.

KEY: optimizer_mode
optoprime

KEY: system_prompt
You are a careful literature-review assistant. Use only the provided source documents. Do not invent unsupported claims. Keep citations/source references faithful to the provided context.


### Result: optim_model_openrouter_prompt_only

,name,process,method,target_model_id,status,returncode,wall_s,log_stdout,log_stderr,error_tail,...,execution_duration_s,promptfoo_duration_s,llm_calls,tool_calls,batch_size,optimizer_mode,param_json,model_param_json,tool_param_json,system_prompt_chars
0,optim_model_openrouter_prompt_only,Direct LLM judge,direct_llm_judge,openrouter/owl-alpha,ok,0,335.684403,scripts\promptfoo_openwebui_eval\results\trace...,scripts\promptfoo_openwebui_eval\results\trace...,,...,110.755964,None,0.0,0.0,1,optoprime,"{""generation_temperature"": 0.2, ""generation_to...","{""generation_temperature"": 0.2, ""generation_to...",,187


### Summary results

,name,process,method,target_model_id,status,returncode,wall_s,log_stdout,log_stderr,error_tail,...,execution_duration_s,promptfoo_duration_s,llm_calls,tool_calls,batch_size,optimizer_mode,param_json,model_param_json,tool_param_json,system_prompt_chars
0,optim_model_openrouter_prompt_only,Direct LLM judge,direct_llm_judge,openrouter/owl-alpha,ok,0,335.684403,scripts\promptfoo_openwebui_eval\results\trace...,scripts\promptfoo_openwebui_eval\results\trace...,,...,110.755964,None,0.0,0.0,1,optoprime,"{""generation_temperature"": 0.2, ""generation_to...","{""generation_temperature"": 0.2, ""generation_to...",,187
